# Emergent Communication Simulation

This notebook runs the full emergent communication experiment:
- Agents communicate about objects via raw byte streams
- A hierarchical encoder/decoder architecture processes the streams
- Entropy-adaptive segmentation discovers emergent 'words'
- We measure whether a compositional vocabulary emerges

Based on the architecture from:
> Neitemeier et al. (2025) *Hierarchical Autoregressive Transformers*

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
from torch.utils.data import DataLoader

from emergent_comm.config import ExperimentConfig
from emergent_comm.environment.world import SignalingGameDataset, collate_trials
from emergent_comm.agents.sender import Sender
from emergent_comm.agents.receiver import Receiver
from emergent_comm.training.trainer import EmergentCommTrainer

## 1. Configuration

In [ ]:
cfg = ExperimentConfig()
cfg.experiment_name = 'baseline_run'

# Reduce for quick smoke test — increase for real experiments
cfg.training.n_steps = 5000
cfg.training.eval_every = 250
cfg.training.batch_size = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Object space size: {cfg.env.n_colors * cfg.env.n_shapes * cfg.env.n_sizes}')

## 2. Build Dataset and DataLoaders

In [ ]:
from emergent_comm.config import EnvConfig

train_dataset = SignalingGameDataset(cfg.env)
val_dataset   = SignalingGameDataset(EnvConfig(seed=cfg.env.seed + 1))

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.training.batch_size,
    collate_fn=collate_trials,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.training.batch_size,
    collate_fn=collate_trials,
    num_workers=0,
)

# Preview a batch
batch = next(iter(train_loader))
print('target_attrs shape   :', batch['target_attrs'].shape)
print('candidate_attrs shape:', batch['candidate_attrs'].shape)
print('target_idx shape     :', batch['target_idx'].shape)

## 3. Build Models

In [ ]:
sender = Sender(
    n_colors=cfg.env.n_colors,
    n_shapes=cfg.env.n_shapes,
    n_sizes=cfg.env.n_sizes,
    bb_cfg=cfg.backbone,
    dec_cfg=cfg.decoder,
    agent_cfg=cfg.agent,
)

receiver = Receiver(
    n_colors=cfg.env.n_colors,
    n_shapes=cfg.env.n_shapes,
    n_sizes=cfg.env.n_sizes,
    enc_cfg=cfg.encoder,
    bb_cfg=cfg.backbone,
)

n_sender   = sum(p.numel() for p in sender.parameters() if p.requires_grad)
n_receiver = sum(p.numel() for p in receiver.parameters() if p.requires_grad)
print(f'Sender parameters  : {n_sender:,}')
print(f'Receiver parameters: {n_receiver:,}')

## 4. Train

In [ ]:
trainer = EmergentCommTrainer(sender, receiver, cfg, device)
history = trainer.train(train_loader, val_loader)

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.steps, history.train_acc, label='train')
axes[0].plot(history.steps, history.val_acc,   label='val')
axes[0].axhline(1 / (cfg.env.n_distractors + 1), color='gray', linestyle='--', label='random')
axes[0].set_title('Communication Accuracy')
axes[0].set_xlabel('Step')
axes[0].legend()

axes[1].plot(history.steps, history.sender_loss,   label='sender (REINFORCE)')
axes[1].plot(history.steps, history.receiver_loss, label='receiver (NLL)')
axes[1].set_title('Losses')
axes[1].set_xlabel('Step')
axes[1].legend()

axes[2].plot(history.steps, history.mean_msg_len)
axes[2].set_title('Mean Message Length (bytes)')
axes[2].set_xlabel('Step')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 6. Save checkpoint for analysis notebook

In [ ]:
import pickle, os
os.makedirs('../../checkpoints', exist_ok=True)

torch.save({'sender': sender.state_dict(), 'receiver': receiver.state_dict()},
           '../../checkpoints/final.pt')

with open('../../checkpoints/history.pkl', 'wb') as f:
    pickle.dump(history, f)

print('Saved.')